# 真实场景 holdout 采集

三个真实场景，一个场景一个 cell。流程和 hard-pair 一样：
先 preview 看 `f_luma/u`，摆好场景后跑场景 cell，按回车开始。

采完自动合并 `data/real_scene.csv`，并用**板上部署权重**算 acc/混淆矩阵（不重训）。
如果真实场景准确率下降，如实写进报告，不要改标签。


In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, 'lib', 'video_setup.py')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'tools'))
import real_scene_notebook as real
print('ROOT =', ROOT)
print('euid =', os.geteuid())


In [ ]:
# 摄像头已由别的 notebook 配好时，把 skip_camera 改成 True
ol, regs, vdma0, vdma1 = real.setup(skip_camera=False, use_manual=True)


## Preview

摆场景时反复跑这一格，看 `f_luma` 是否在目标带、`u` 是否合理。


In [ ]:
real.preview(regs)


## Scene 0：真实暗角/低光

目标 `f_luma` 4~40；画面里放有纹理的物体，不要遮黑镜头。


In [ ]:
rows0 = real.collect_scene(regs, 0, n=40, wait=8)


## Scene 1：真实窗边逆光

目标 `f_luma` 40~105；窗/灯进入画面，主体背光。


In [ ]:
rows1 = real.collect_scene(regs, 1, n=40, wait=8)


## Scene 2：真实室内正常光

目标 `f_luma` 80~145；有纹理的桌面/墙面，避免窗/灯直射。


In [ ]:
rows2 = real.collect_scene(regs, 2, n=40, wait=8)


## 合并 + 评估

`eval_holdout.py` 会打印纯 MLP 和 MLP+四象限的 acc/混淆矩阵。


In [ ]:
real.merge()
real.evaluate()


## 收尾

停 VDMA、释放 overlay。


In [ ]:
real.cleanup(ol, vdma0, vdma1)
